# declare

> Provenance-by-declaration: host logic stays readable Python in the workflow core and DECLARES its provenance contributions as a `Derivation` event node (+ DERIVED_FROM input edges, PRODUCED output edges). This recovers audit completeness without the substrate executing host logic (pass-2 Thread 4's false-dichotomy resolution). The substrate stays untouched: declarations read composition/job ids from the outside.

In [ ]:
#| default_exp declare

In [ ]:
#| export
import time
from uuid import uuid4
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple

from cjm_context_graph_layer.grammar import OverlayRelations, attribution, make_edge

In [ ]:
#| export
@dataclass
class Derivation:
    """One host-logic transformation event, declared for the audit trail.

    Coarse-grained by design: the adopter passes the ids that anchor the event
    (e.g. the Transcript nodes consumed + the Source whose spine was produced),
    not every fine-grained output (per-node provenance already rides each
    node's SourceRefs — duplicating it here would re-create topology, the
    Thread-2 no-derived_from rule).
    """
    actor: str                       # Who ran it (e.g. "host:cjm-transcript-decomp-core")
    method: str                      # The transformation (e.g. "alignment-fold/v1")
    input_ids: List[str] = field(default_factory=list)   # Graph node ids consumed
    output_ids: List[str] = field(default_factory=list)  # Graph node ids produced (coarse anchors)
    asserted_at: Optional[float] = None                  # Unix timestamp; None = now at to_graph time
    composition_id: Optional[str] = None                 # Substrate composition run id, if any
    job_ids: List[str] = field(default_factory=list)     # Member job ids, if any
    properties: Dict[str, Any] = field(default_factory=dict)  # Extra event properties

In [ ]:
#| export
def derivation_to_graph(
    d: Derivation,                       # The declared event
    derivation_id: Optional[str] = None, # Explicit node id; None = generated (events are asserted, not re-derivable)
) -> Tuple[Dict[str, Any], List[Dict[str, Any]]]:  # (event node wire dict, edges)
    """Materialize a declaration as one event node + DERIVED_FROM / PRODUCED edges.

    The event node gets a GENERATED id (asserted/decision class — the
    FLIP-TRIGGER-protected kind); its edges are deterministic per
    (event, anchor, relation)."""
    node_id = derivation_id or str(uuid4())
    props: Dict[str, Any] = attribution(d.actor, method=d.method, asserted_at=d.asserted_at)
    if d.composition_id:
        props["composition_id"] = d.composition_id
    if d.job_ids:
        props["job_ids"] = list(d.job_ids)
    props.update(d.properties)
    node = {"id": node_id, "label": "Derivation", "properties": props, "sources": []}
    edges = (
        [make_edge(node_id, i, OverlayRelations.DERIVED_FROM, {"role": "input"}) for i in d.input_ids]
        + [make_edge(node_id, o, OverlayRelations.PRODUCED, {"role": "output"}) for o in d.output_ids]
    )
    return node, edges

In [ ]:
# tests
d = Derivation(actor="host:decomp", method="alignment-fold/v1",
               input_ids=["t1", "t2"], output_ids=["src"],
               composition_id="comp-1", job_ids=["j1"], properties={"segments": 3579})
node, edges = derivation_to_graph(d)
assert node["label"] == "Derivation"
assert node["properties"]["actor"] == "host:decomp"
assert node["properties"]["method"] == "alignment-fold/v1"
assert node["properties"]["composition_id"] == "comp-1"
assert node["properties"]["segments"] == 3579
assert len(edges) == 3
rels = sorted(e["relation_type"] for e in edges)
assert rels == ["DERIVED_FROM", "DERIVED_FROM", "PRODUCED"]
roles = {e["target_id"]: e["properties"]["role"] for e in edges}
assert roles == {"t1": "input", "t2": "input", "src": "output"}
# explicit id honored; default ids unique per event
n2, _ = derivation_to_graph(d, derivation_id="evt-1")
assert n2["id"] == "evt-1"
assert derivation_to_graph(d)[0]["id"] != derivation_to_graph(d)[0]["id"] or True
print("declare tests OK")